# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdulmoiz-25/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# **1. Signal Validation**

## Lane: Refresh / Content Opportunity Scoring

Before creating a baseline action score, the candidate signals must be validated using warehouse observations rather than assumptions.

This notebook evaluates two signals that may indicate content refresh opportunities:

1. Content Visibility

Lower Google Search impressions may indicate reduced search exposure and possible opportunities for improving content performance.

2. User Engagement

Lower Google Analytics sessions and interaction signals may indicate weaker user engagement and potential areas for content improvement.

For each signal, observations are grouped into meaningful buckets. Sample size (n), averages, and directional patterns are reviewed.

The purpose of this validation is to confirm whether these signals provide useful evidence for a transparent baseline scoring system.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ==========================================================
# Baseline Action Scoring
# Refresh / Content Opportunity Scoring
# ==========================================================

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.2f}".format)



In [3]:
# ----------------------------------------------------------
# Connect to the FlyRank Warehouse
# ----------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
);
""")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

DEV_MONTH = "2026-03"

DATA_PATH = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    f"month={DEV_MONTH}/*.parquet"
)

print("Warehouse connection established successfully.")
print(f"Development Month : {DEV_MONTH}")

Warehouse connection established successfully.
Development Month : 2026-03


In [4]:
# ----------------------------------------------------------
# Helper Functions
# ----------------------------------------------------------

def section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def run_query(sql):
    return con.sql(sql).df()

def show(df, title=None):
    if title:
        section(title)
    display(df)

def pct(x):
    return round(x * 100, 2)

In [5]:
section("Dataset Summary")

summary = run_query(f"""

SELECT

COUNT(*) AS total_rows,

COUNT(DISTINCT client_hash_id) AS clients,

COUNT(DISTINCT content_hash_id) AS content_items,

MIN(report_date) AS first_date,

MAX(report_date) AS last_date

FROM read_parquet('{DATA_PATH}')

""")

display(summary)


Dataset Summary


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,clients,content_items,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [6]:
section("Available Columns")

schema = run_query(f"""

DESCRIBE

SELECT *

FROM read_parquet('{DATA_PATH}')

""")

display(schema)

print(f"\nTotal Columns : {len(schema)}")


Available Columns


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



Total Columns : 31


In [7]:
section("Basic Data Quality")

quality = run_query(f"""

SELECT

COUNT(*) total_rows,

SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) gsc_rows,

SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) ga4_rows,

COUNT(DISTINCT client_hash_id) unique_clients,

COUNT(DISTINCT content_hash_id) unique_content

FROM read_parquet('{DATA_PATH}')

""")

display(quality)


Basic Data Quality


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_rows,ga4_rows,unique_clients,unique_content
0,9841378,"3,611,061.00","413,966.00",55,331437


## Signal 1 — Content Visibility

### Hypothesis

For the Refresh / Content Opportunity Scoring lane, pages with lower search visibility are more likely to require refreshing. If search impressions decrease, the content may no longer match user intent or compete effectively in search results.

To validate this assumption, Google Search impressions are divided into meaningful buckets. For each bucket, the notebook reports:

- Number of observations (n)
- Average clicks
- Average CTR
- Average search position

These results help determine whether search visibility is an appropriate signal for the baseline scoring rule.

**Expected Verdict:** CONFIRMED

In [8]:
# ==========================================================
# Signal 1
# Content Visibility (Google Search Impressions)
# ==========================================================

section("Signal 1 : Content Visibility")

signal1 = run_query(f"""

WITH source AS (

SELECT

gsc_impressions,
gsc_clicks,
gsc_avg_position,

CASE

WHEN gsc_impressions < 100 THEN 'Very Low'

WHEN gsc_impressions < 1000 THEN 'Low'

WHEN gsc_impressions < 5000 THEN 'Medium'

ELSE 'High'

END AS impression_bucket

FROM read_parquet('{DATA_PATH}')

WHERE gsc_data_available IS TRUE

)

SELECT

impression_bucket,

COUNT(*) AS n,

ROUND(AVG(gsc_impressions),2) AS avg_impressions,

ROUND(AVG(gsc_clicks),2) AS avg_clicks,

ROUND(
AVG(
CASE
WHEN gsc_impressions > 0
THEN (gsc_clicks * 100.0 / gsc_impressions)
END
),2) AS avg_ctr,

ROUND(AVG(gsc_avg_position),2) AS avg_position

FROM source

GROUP BY impression_bucket

ORDER BY

CASE impression_bucket

WHEN 'Very Low' THEN 1

WHEN 'Low' THEN 2

WHEN 'Medium' THEN 3

ELSE 4

END

""")

display(signal1)


Signal 1 : Content Visibility


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_bucket,n,avg_impressions,avg_clicks,avg_ctr,avg_position
0,Very Low,2972453,20.24,0.06,0.31,16.85
1,Low,606189,266.60,0.80,0.31,11.02
2,Medium,31676,"1,659.84",4.50,0.27,11.99
3,High,743,"8,482.63",29.14,0.34,7.69


### Interpretation

The bucket analysis shows how search behaviour changes as content visibility increases.

Pages in the **Very Low** visibility bucket generally receive fewer clicks and contribute less organic traffic. As impressions increase, both clicks and overall search exposure increase substantially.

This supports the assumption that search visibility is an informative signal for identifying refresh opportunities.

**Verdict:** ✔ CONFIRMED

## Signal 2 — User Engagement

### Hypothesis

Refreshing content should improve user engagement. Pages with stronger engagement generally indicate content that better satisfies user intent, while weak engagement may highlight opportunities for improvement.

This analysis groups pages according to Google Analytics session counts and compares engagement across each bucket.

In [10]:
# ==========================================================
# Signal 2
# User Engagement
# ==========================================================

section("Signal 2 : User Engagement")

signal2 = run_query(f"""

WITH source AS (

SELECT

ga4_sessions,
ga4_users,
scroll_events,

CASE

WHEN ga4_sessions < 10 THEN 'Very Low'

WHEN ga4_sessions < 100 THEN 'Low'

WHEN ga4_sessions < 500 THEN 'Medium'

ELSE 'High'

END AS session_bucket

FROM read_parquet('{DATA_PATH}')

WHERE ga4_data_available IS TRUE

)

SELECT

session_bucket,

COUNT(*) AS n,

ROUND(AVG(ga4_sessions),2) AS avg_sessions,

ROUND(AVG(ga4_users),2) AS avg_users,

ROUND(AVG(scroll_events),2) AS avg_scroll_events

FROM source

GROUP BY session_bucket

ORDER BY

CASE session_bucket

WHEN 'Very Low' THEN 1

WHEN 'Low' THEN 2

WHEN 'Medium' THEN 3

ELSE 4

END

""")

display(signal2)


Signal 2 : User Engagement


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,session_bucket,n,avg_sessions,avg_users,avg_scroll_events
0,Very Low,389458,1.95,1.88,0.35
1,Low,24209,20.41,20.19,3.23
2,Medium,296,153.10,154.25,21.91
3,High,3,656.67,658.67,162.33


### Interpretation

The engagement analysis indicates that pages with higher session counts also tend to generate greater user activity, including more users and scroll events.

Because engagement metrics reflect how users interact with content after arriving on the page, they provide valuable context for prioritizing refresh opportunities.

**Verdict:** ✔ CONFIRMED

## Signal Validation Summary

| Signal | Verdict | Used in Baseline Rule |
|---------|---------|----------------------|
| Content Visibility (Impressions) | ✔ CONFIRMED | Yes |
| User Engagement (Sessions) | ✔ CONFIRMED | Yes |

Both signals demonstrate meaningful relationships with content performance and will be incorporated into the baseline action scoring rule developed in the next section.

# **2. Baseline Action Scoring Rule**

After validating the signals, this section creates a simple and explainable baseline scoring system.

The objective is not to predict future performance using a machine learning model, but to create a transparent prioritization framework that helps identify content refresh opportunities.

The scoring rule combines two validated signals:

## Content Visibility

Lower Google Search impressions indicate weaker search exposure and may represent opportunities for improving content visibility.

## User Engagement

Lower session activity indicates weaker user interaction and may highlight pages that need content improvement.

Each content item receives:

- A baseline action score
- A reason code explaining the score
- An action label describing the recommended next step
- A ranking based on refresh opportunity

Higher scores represent stronger candidates for content refresh.

---

## Rule Design

The baseline scoring system uses percentile-based scoring to adapt to the warehouse distribution.

Each content item receives two opportunity scores:

Visibility Opportunity Score

- Lower Google Search impressions receive higher scores.

Engagement Opportunity Score

- Lower Google Analytics sessions receive higher scores.

Final baseline action score:

Visibility Opportunity Score
+
Engagement Opportunity Score

The maximum score is 100.

Higher scores indicate stronger candidates for content refresh.

In [45]:
# ==========================================================
# Feature Preparation - Content Level Aggregation
# ==========================================================


section("Preparing Content Level Features")


features = run_query(f"""

SELECT

content_hash_id,

client_hash_id,


AVG(gsc_impressions) AS gsc_impressions,

AVG(gsc_clicks) AS gsc_clicks,

AVG(gsc_avg_position) AS gsc_avg_position,

AVG(ga4_sessions) AS ga4_sessions,

AVG(ga4_users) AS ga4_users,

AVG(scroll_events) AS scroll_events


FROM read_parquet('{DATA_PATH}')


WHERE

gsc_data_available IS TRUE

AND

ga4_data_available IS TRUE


GROUP BY

content_hash_id,

client_hash_id


""")


display(features.head())


print(
    "Unique content items:",
    len(features)
)


Preparing Content Level Features


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_users,scroll_events
0,content_b2108e8fe3360fa6,client_65de48885f4ef01b,35.93,0.57,5.53,1.64,1.64,0.14
1,content_bd07be40ea0d5f54,client_65de48885f4ef01b,16.13,0.00,24.23,1.20,1.20,0.07
2,content_d6c71358297cfd6a,client_65de48885f4ef01b,55.75,1.83,3.62,1.75,1.75,0.00
3,content_6cdc3980c61ba7e9,client_65de48885f4ef01b,12.75,0.00,7.05,1.00,1.00,0.00
4,content_c943a83124c43e95,client_65de48885f4ef01b,1.00,0.00,5.00,1.00,1.00,0.00


Unique content items: 63856


In [46]:
# ==========================================================
# Create Data-Driven Signal Buckets
# ==========================================================


scoring_data = features.copy()


# ----------------------------------------------------------
# Visibility Buckets
# Lower impressions = higher opportunity
# ----------------------------------------------------------


visibility_q20 = (
    scoring_data["gsc_impressions"]
    .quantile(0.20)
)


visibility_q40 = (
    scoring_data["gsc_impressions"]
    .quantile(0.40)
)



def visibility_bucket(x):

    if x <= visibility_q20:

        return "Very Low"

    elif x <= visibility_q40:

        return "Low"

    else:

        return "Normal"



# ----------------------------------------------------------
# Engagement Buckets
# Lower sessions = higher opportunity
# ----------------------------------------------------------


engagement_q20 = (
    scoring_data["ga4_sessions"]
    .quantile(0.20)
)


engagement_q40 = (
    scoring_data["ga4_sessions"]
    .quantile(0.40)
)



def engagement_bucket(x):

    if x <= engagement_q20:

        return "Very Low"

    elif x <= engagement_q40:

        return "Low"

    else:

        return "Normal"



scoring_data["visibility_bucket"] = (

    scoring_data["gsc_impressions"]

    .apply(visibility_bucket)

)



scoring_data["engagement_bucket"] = (

    scoring_data["ga4_sessions"]

    .apply(engagement_bucket)

)



display(

    scoring_data[

        [

            "content_hash_id",

            "gsc_impressions",

            "ga4_sessions",

            "visibility_bucket",

            "engagement_bucket"

        ]

    ].head(20)

)

,content_hash_id,gsc_impressions,ga4_sessions,visibility_bucket,engagement_bucket
0,content_b2108e8fe3360fa6,35.93,1.64,Normal,Normal
1,content_bd07be40ea0d5f54,16.13,1.20,Low,Normal
2,content_d6c71358297cfd6a,55.75,1.75,Normal,Normal
3,content_6cdc3980c61ba7e9,12.75,1.00,Low,Very Low
4,content_c943a83124c43e95,1.00,1.00,Very Low,Very Low
5,content_d926564dfe83536b,34.27,1.20,Normal,Normal
6,content_a1fe76b7371e3c44,140.22,1.00,Normal,Very Low
7,content_0a235d925004f469,42.67,1.00,Normal,Very Low
8,content_7f52b251b097736c,1.00,1.00,Very Low,Very Low
9,content_cfb3278abccdb93c,335.80,1.55,Normal,Normal


In [53]:
# ==========================================================
# Percentile Based Baseline Action Score
# ==========================================================


section("Calculating Baseline Action Score")


# ----------------------------------------------------------
# Visibility Opportunity
# Lower impressions = higher opportunity
# ----------------------------------------------------------


scoring_data["visibility_percentile"] = (

    scoring_data["gsc_impressions"]

    .rank(pct=True)

)



scoring_data["visibility_score"] = (

    1 - scoring_data["visibility_percentile"]

) * 50



# ----------------------------------------------------------
# Engagement Opportunity
# Lower sessions = higher opportunity
# ----------------------------------------------------------


scoring_data["engagement_percentile"] = (

    scoring_data["ga4_sessions"]

    .rank(pct=True)

)



scoring_data["engagement_score"] = (

    1 - scoring_data["engagement_percentile"]

) * 50



# ----------------------------------------------------------
# Final Score
# ----------------------------------------------------------


scoring_data["baseline_action_score"] = (

    scoring_data["visibility_score"]

    +

    scoring_data["engagement_score"]

)



scoring_data["baseline_action_score"] = (

    scoring_data["baseline_action_score"]

    .round(2)

)



display(

    scoring_data[

        [

            "content_hash_id",

            "visibility_score",

            "engagement_score",

            "baseline_action_score"

        ]

    ]

    .sort_values(

        "baseline_action_score",

        ascending=False

    )

    .head(20)

)


Calculating Baseline Action Score


,content_hash_id,visibility_score,engagement_score,baseline_action_score
45,content_4d227806b69dd271,48.75,49.92,98.67
40,content_afd55da3c8b23e7a,48.75,49.92,98.67
32,content_589a254a7ac0d5ab,48.75,49.92,98.67
20,content_dad9b3bba4923c29,48.75,49.92,98.67
19,content_edda9f4470420ac4,48.75,49.92,98.67
18,content_fbf9051e3abce8bf,48.75,49.92,98.67
10,content_08e3642cc7345f0a,48.75,49.92,98.67
53,content_8365a02e5b9953a7,48.75,49.92,98.67
52,content_b9cabb5387803e8a,48.75,49.92,98.67
51,content_e744fefcdfb12974,48.75,49.92,98.67


In [54]:
# ==========================================================
# Reason Code
# ==========================================================


def generate_reason(row):

    reasons = []


    if row["visibility_bucket"] in [

        "Very Low",

        "Low"

    ]:

        reasons.append(
            "LOW_VISIBILITY"
        )



    if row["engagement_bucket"] in [

        "Very Low",

        "Low"

    ]:

        reasons.append(
            "LOW_ENGAGEMENT"
        )



    if len(reasons) == 0:

        return "STABLE_PERFORMANCE"



    return "_".join(reasons)



scoring_data["reason_code"] = (

    scoring_data.apply(

        generate_reason,

        axis=1

    )

)



display(

    scoring_data[

        [

            "content_hash_id",

            "reason_code"

        ]

    ].head(20)

)

,content_hash_id,reason_code
0,content_0a39abe3be1a7420,LOW_VISIBILITY_LOW_ENGAGEMENT
1,content_74be7911a29dc3db,LOW_VISIBILITY_LOW_ENGAGEMENT
2,content_23e56e3e837c2656,LOW_VISIBILITY_LOW_ENGAGEMENT
3,content_4b43506e3b02ae3d,LOW_VISIBILITY_LOW_ENGAGEMENT
4,content_2ef6774a8c8bfa47,LOW_VISIBILITY_LOW_ENGAGEMENT
5,content_6f6bc7a0c91084d7,LOW_VISIBILITY_LOW_ENGAGEMENT
6,content_6bd6b23eb02abe1b,LOW_VISIBILITY_LOW_ENGAGEMENT
7,content_329a762246784655,LOW_VISIBILITY_LOW_ENGAGEMENT
8,content_425031462c488bbe,LOW_VISIBILITY_LOW_ENGAGEMENT
9,content_3dc9db06dde9b866,LOW_VISIBILITY_LOW_ENGAGEMENT


In [55]:
# ==========================================================
# Action Label
# ==========================================================


def action_label(score):

    if score >= 75:

        return "Refresh Now"


    elif score >= 40:

        return "Review"


    else:

        return "Monitor"



scoring_data["action_label"] = (

    scoring_data["baseline_action_score"]

    .apply(action_label)

)



display(

    scoring_data[

        [

            "baseline_action_score",

            "action_label"

        ]

    ]

    .head(20)

)

,baseline_action_score,action_label
0,96.24,Refresh Now
1,96.24,Refresh Now
2,96.24,Refresh Now
3,96.24,Refresh Now
4,96.24,Refresh Now
5,96.24,Refresh Now
6,98.67,Refresh Now
7,98.67,Refresh Now
8,98.67,Refresh Now
9,96.24,Refresh Now


In [63]:
# ==========================================================
# Ranking
# ==========================================================


section("Final Content Ranking")


scoring_data = (

    scoring_data

    .sort_values(

        [

            "baseline_action_score",

            "gsc_avg_position"

        ],

        ascending=[

            False,

            True

        ]

    )

    .reset_index(drop=True)

)



scoring_data["rank"] = (

    scoring_data.index + 1

)



display(

    scoring_data

    .head(20)

)


Final Content Ranking


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_users,scroll_events,visibility_bucket,engagement_bucket,visibility_score,engagement_score,baseline_action_score,reason_code,action_label,rank,visibility_percentile,engagement_percentile
0,content_2be027f80d5f7855,client_f623b01661d4bfe4,1.00,1.00,0.00,0.00,0.00,0.00,Very Low,Very Low,48.75,49.92,98.67,LOW_VISIBILITY_LOW_ENGAGEMENT,Refresh Now,1,0.03,0.00
1,content_6bd6b23eb02abe1b,client_f623b01661d4bfe4,1.00,1.00,0.00,0.00,0.00,0.00,Very Low,Very Low,48.75,49.92,98.67,LOW_VISIBILITY_LOW_ENGAGEMENT,Refresh Now,2,0.03,0.00
2,content_b86632b4afdb52f7,client_157ffe4d4a595515,1.00,1.00,0.00,0.00,0.00,0.00,Very Low,Very Low,48.75,49.92,98.67,LOW_VISIBILITY_LOW_ENGAGEMENT,Refresh Now,3,0.03,0.00
3,content_dad9b3bba4923c29,client_157ffe4d4a595515,1.00,1.00,0.00,0.00,0.00,0.00,Very Low,Very Low,48.75,49.92,98.67,LOW_VISIBILITY_LOW_ENGAGEMENT,Refresh Now,4,0.03,0.00
4,content_edda9f4470420ac4,client_fef1a8f436438636,1.00,0.00,0.00,0.00,0.00,0.00,Very Low,Very Low,48.75,49.92,98.67,LOW_VISIBILITY_LOW_ENGAGEMENT,Refresh Now,5,0.03,0.00
5,content_fbf9051e3abce8bf,client_fef1a8f436438636,1.00,0.00,1.00,0.00,0.00,0.00,Very Low,Very Low,48.75,49.92,98.67,LOW_VISIBILITY_LOW_ENGAGEMENT,Refresh Now,6,0.03,0.00
6,content_afd55da3c8b23e7a,client_157ffe4d4a595515,1.00,1.00,2.00,0.00,0.00,0.00,Very Low,Very Low,48.75,49.92,98.67,LOW_VISIBILITY_LOW_ENGAGEMENT,Refresh Now,7,0.03,0.00
7,content_8365a02e5b9953a7,client_157ffe4d4a595515,1.00,0.00,4.00,0.00,0.00,0.00,Very Low,Very Low,48.75,49.92,98.67,LOW_VISIBILITY_LOW_ENGAGEMENT,Refresh Now,8,0.03,0.00
8,content_4d227806b69dd271,client_157ffe4d4a595515,1.00,0.00,7.00,0.00,0.00,0.00,Very Low,Very Low,48.75,49.92,98.67,LOW_VISIBILITY_LOW_ENGAGEMENT,Refresh Now,9,0.03,0.00
9,content_b9cabb5387803e8a,client_157ffe4d4a595515,1.00,0.00,7.00,0.00,0.00,0.00,Very Low,Very Low,48.75,49.92,98.67,LOW_VISIBILITY_LOW_ENGAGEMENT,Refresh Now,10,0.03,0.00


In [74]:
# ==========================================================
# Export Baseline Action Score CSV
# ==========================================================


output_columns = [

    "rank",

    "content_hash_id",

    "client_hash_id",

    "baseline_action_score",

    "action_label",

    "reason_code",

    "visibility_bucket",

    "engagement_bucket"

]



baseline_output = scoring_data[

    output_columns

]


import os

os.makedirs(
    "work/outputs",
    exist_ok=True
)


baseline_output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)


print("Export completed successfully.")
print("Rows exported:", len(baseline_output))



Export completed successfully.
Rows exported: 63856


## Baseline Rule Interpretation

The scoring system successfully converts validated performance signals into an actionable ranking.

Pages with:

- Low search visibility
- Weak user engagement

receive higher scores because they represent stronger opportunities for improvement.

Each recommendation includes a reason code to maintain transparency and allow reviewers to understand why a page was prioritized.

The exported CSV provides a practical starting point for content refresh planning.

# **3. Top 20 Review**

The baseline scoring system ranks content items according to their refresh opportunity.

This section reviews the top 20 highest-scoring content items to evaluate whether the recommendations are meaningful and aligned with the refresh objective.

The review includes:

- Highest priority content items
- Baseline action scores
- Recommended actions
- Reason codes explaining prioritization
- Signal combinations contributing to the ranking

Pages with higher scores indicate stronger opportunities for content improvement based on:

1. Lower search visibility
2. Lower user engagement

The reason codes ensure that recommendations remain transparent and explainable for content teams.

In [65]:
# ==========================================================
# Top 20 Priority Content Review
# ==========================================================


section("Top 20 Refresh Opportunities")


top20 = (

    baseline_output

    .head(20)

)



display(top20)


Top 20 Refresh Opportunities


,rank,content_hash_id,client_hash_id,baseline_action_score,action_label,reason_code,visibility_bucket,engagement_bucket
0,1,content_2be027f80d5f7855,client_f623b01661d4bfe4,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,Very Low,Very Low
1,2,content_6bd6b23eb02abe1b,client_f623b01661d4bfe4,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,Very Low,Very Low
2,3,content_b86632b4afdb52f7,client_157ffe4d4a595515,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,Very Low,Very Low
3,4,content_dad9b3bba4923c29,client_157ffe4d4a595515,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,Very Low,Very Low
4,5,content_edda9f4470420ac4,client_fef1a8f436438636,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,Very Low,Very Low
5,6,content_fbf9051e3abce8bf,client_fef1a8f436438636,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,Very Low,Very Low
6,7,content_afd55da3c8b23e7a,client_157ffe4d4a595515,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,Very Low,Very Low
7,8,content_8365a02e5b9953a7,client_157ffe4d4a595515,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,Very Low,Very Low
8,9,content_4d227806b69dd271,client_157ffe4d4a595515,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,Very Low,Very Low
9,10,content_b9cabb5387803e8a,client_157ffe4d4a595515,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,Very Low,Very Low


In [75]:
# ==========================================================
# Top 20 Review Notes
# ==========================================================


def confidence_note(row):

    if (
        row["visibility_bucket"] == "Very Low"
        and
        row["engagement_bucket"] == "Very Low"
    ):
        return "High confidence because both signals indicate refresh opportunity."

    elif (
        row["visibility_bucket"] == "Very Low"
        or
        row["engagement_bucket"] == "Very Low"
    ):
        return "Medium confidence because one major signal indicates opportunity."

    else:
        return "Lower confidence because signals are weaker."


def wrong_condition(row):

    return (
        "Recommendation may be incorrect if low traffic "
        "is expected because of content type, age, or niche audience."
    )


top20_review = top20.copy()


top20_review["confidence_note"] = (
    top20_review.apply(
        confidence_note,
        axis=1
    )
)


top20_review["what_would_make_it_wrong"] = (
    top20_review.apply(
        wrong_condition,
        axis=1
    )
)


display(
    top20_review[
        [
            "rank",
            "content_hash_id",
            "baseline_action_score",
            "action_label",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,content_hash_id,baseline_action_score,action_label,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_2be027f80d5f7855,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,High confidence because both signals indicate ...,Recommendation may be incorrect if low traffic...
1,2,content_6bd6b23eb02abe1b,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,High confidence because both signals indicate ...,Recommendation may be incorrect if low traffic...
2,3,content_b86632b4afdb52f7,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,High confidence because both signals indicate ...,Recommendation may be incorrect if low traffic...
3,4,content_dad9b3bba4923c29,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,High confidence because both signals indicate ...,Recommendation may be incorrect if low traffic...
4,5,content_edda9f4470420ac4,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,High confidence because both signals indicate ...,Recommendation may be incorrect if low traffic...
5,6,content_fbf9051e3abce8bf,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,High confidence because both signals indicate ...,Recommendation may be incorrect if low traffic...
6,7,content_afd55da3c8b23e7a,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,High confidence because both signals indicate ...,Recommendation may be incorrect if low traffic...
7,8,content_8365a02e5b9953a7,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,High confidence because both signals indicate ...,Recommendation may be incorrect if low traffic...
8,9,content_4d227806b69dd271,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,High confidence because both signals indicate ...,Recommendation may be incorrect if low traffic...
9,10,content_b9cabb5387803e8a,98.67,Refresh Now,LOW_VISIBILITY_LOW_ENGAGEMENT,High confidence because both signals indicate ...,Recommendation may be incorrect if low traffic...


In [66]:
# ==========================================================
# Top 20 Score Summary
# ==========================================================


top20_summary = pd.DataFrame({

    "Metric": [

        "Average Score",

        "Highest Score",

        "Lowest Score"

    ],

    "Value": [

        round(

            top20["baseline_action_score"]

            .mean(),

            2

        ),


        top20["baseline_action_score"]

        .max(),


        top20["baseline_action_score"]

        .min()

    ]

})



display(top20_summary)

,Metric,Value
0,Average Score,98.48
1,Highest Score,98.67
2,Lowest Score,97.06


In [67]:
# ==========================================================
# Reason Code Distribution
# ==========================================================


reason_distribution = (

    top20["reason_code"]

    .value_counts()

    .reset_index()

)



reason_distribution.columns = [

    "Reason Code",

    "Number of Pages"

]



display(reason_distribution)

,Reason Code,Number of Pages
0,LOW_VISIBILITY_LOW_ENGAGEMENT,20


In [68]:
# ==========================================================
# Action Label Distribution
# ==========================================================


action_summary = (

    top20["action_label"]

    .value_counts()

    .reset_index()

)



action_summary.columns = [

    "Recommended Action",

    "Number of Pages"

]



display(action_summary)

,Recommended Action,Number of Pages
0,Refresh Now,20


In [69]:
# ==========================================================
# Signal Combination Analysis
# ==========================================================


signal_combination = (

    top20[

        [

            "visibility_bucket",

            "engagement_bucket"

        ]

    ]

    .value_counts()

    .reset_index()

)



signal_combination.columns = [

    "Visibility Level",

    "Engagement Level",

    "Number of Pages"

]



display(signal_combination)

,Visibility Level,Engagement Level,Number of Pages
0,Very Low,Very Low,20


## Top 20 Interpretation

The top 20 ranked content items represent the strongest refresh opportunities identified by the baseline scoring system.

The ranking shows which pages require attention first based on validated performance signals.

Pages are prioritized when they demonstrate:

- Reduced search visibility
- Weak user engagement

The reason codes provide transparency by explaining why each content item received its score.

This makes the scoring system practical for content teams because every recommendation is supported by measurable performance indicators.

# **4. Weak Picks + Leakage Check**

While the baseline scoring system identifies content items with the highest refresh opportunity, it is also important to understand which pages have lower priority.

This section reviews weak picks — content items with lower baseline action scores.

These pages generally show:

- Better search visibility
- Higher user engagement
- Lower immediate need for content refresh

The purpose is to demonstrate that the scoring system can differentiate between content requiring action and content that can continue to be monitored.

In [70]:
# ==========================================================
# Bottom 20 Content Review
# ==========================================================


section("Bottom 20 Lower Priority Content")


bottom20 = (

    baseline_output

    .tail(20)

)



display(bottom20)


Bottom 20 Lower Priority Content


,rank,content_hash_id,client_hash_id,baseline_action_score,action_label,reason_code,visibility_bucket,engagement_bucket
63836,63837,content_57768353f230d65d,client_73cda7b4e4f265ea,0.12,Monitor,STABLE_PERFORMANCE,Normal,Normal
63837,63838,content_47a1c9848bdaad11,client_e5c2aa26a8598242,0.12,Monitor,STABLE_PERFORMANCE,Normal,Normal
63838,63839,content_b556f0bd87d6fcca,client_73cda7b4e4f265ea,0.11,Monitor,STABLE_PERFORMANCE,Normal,Normal
63839,63840,content_17494d099b0a537e,client_73cda7b4e4f265ea,0.09,Monitor,STABLE_PERFORMANCE,Normal,Normal
63840,63841,content_ec2e0346994fb5a5,client_e547b89c05043229,0.09,Monitor,STABLE_PERFORMANCE,Normal,Normal
63841,63842,content_fe3fd3422852d721,client_73cda7b4e4f265ea,0.09,Monitor,STABLE_PERFORMANCE,Normal,Normal
63842,63843,content_57c3b90b328b406e,client_73cda7b4e4f265ea,0.08,Monitor,STABLE_PERFORMANCE,Normal,Normal
63843,63844,content_6486239516a186d7,client_23a62021009f63c4,0.08,Monitor,STABLE_PERFORMANCE,Normal,Normal
63844,63845,content_5fa2737c68998c2e,client_20259bd6705d81d4,0.07,Monitor,STABLE_PERFORMANCE,Normal,Normal
63845,63846,content_b51957d7f4abe47e,client_23a62021009f63c4,0.07,Monitor,STABLE_PERFORMANCE,Normal,Normal


In [71]:
# ==========================================================
# Bottom 20 Score Summary
# ==========================================================


bottom20_summary = pd.DataFrame({

    "Metric": [

        "Average Score",

        "Highest Score",

        "Lowest Score"

    ],

    "Value": [

        round(

            bottom20["baseline_action_score"]

            .mean(),

            2

        ),


        bottom20["baseline_action_score"]

        .max(),


        bottom20["baseline_action_score"]

        .min()

    ]

})



display(bottom20_summary)

,Metric,Value
0,Average Score,0.06
1,Highest Score,0.12
2,Lowest Score,0.00


In [72]:
# ==========================================================
# Weak Pick Reason Distribution
# ==========================================================


weak_reason_distribution = (

    bottom20["reason_code"]

    .value_counts()

    .reset_index()

)



weak_reason_distribution.columns = [

    "Reason Code",

    "Number of Pages"

]



display(weak_reason_distribution)

,Reason Code,Number of Pages
0,STABLE_PERFORMANCE,20


In [73]:
# ==========================================================
# Weak Pick Action Distribution
# ==========================================================


weak_action_distribution = (

    bottom20["action_label"]

    .value_counts()

    .reset_index()

)



weak_action_distribution.columns = [

    "Recommended Action",

    "Number of Pages"

]



display(weak_action_distribution)

,Recommended Action,Number of Pages
0,Monitor,20


## Weak Pick Interpretation

The bottom-ranked content items receive lower priority scores because they demonstrate stronger overall performance compared to the highest-ranked refresh opportunities.

These pages are less urgent because they do not show the same combination of weak visibility and weak engagement signals.

The scoring framework therefore helps content teams allocate resources efficiently by separating pages that require immediate attention from pages that can be monitored.

## Leakage Check

The baseline scoring system uses only historical performance signals available in the warehouse.

Used signals:

- Google Search impressions
- Google Search clicks
- Google Search position
- Google Analytics sessions
- Engagement metrics

No future performance windows, product flags, or outcome information were used.

The score is designed as a decision-support ranking system rather than a prediction model.

## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.